import libraries
prepare the data
split the data into:
    train
    validation
    test
add the model 
add randomized search parameters
save the best hyperparameters for the model
fit the tuned model to the training data
see the evaluation resutls on training dataset
check it on vlaidvalidation dataset 
- if perform well 
check it on test dataset
- if not: change the model hyperparameters and once satisfied 
check it on test dataset
plot the feature importance


In [1]:
# import necessary libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

#preprocessing 
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
#modelselection
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV,train_test_split, KFold
#model 
from sklearn.ensemble import RandomForestClassifier
#evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("../data/diseaseB.csv")
df.head()


,age,bmi,blood_pressure,glucose_level,family_history,physical_activity_level,has_disease
0,19,25.2,133,111,1,moderate,1
1,64,38.5,109,106,1,moderate,0
2,58,34.3,133,100,0,moderate,0
3,21,26.8,144,118,0,moderate,0
4,79,29.5,161,132,0,low,1


SPLIT FIRST , THEN ENCODE
Split will be around train-validation-test, to do so we have to do 2 steps split first 70% training and 30% temporayr and then that 30 goes to validataion and test 

In [3]:

# features: columns except target
X = df.drop("has_disease",axis=1)
# target: column you want to classify
y = df["has_disease"]

In [4]:
# FIRST CUT: Split out the Training set (80%)

X_train, X_temp, y_train,y_temp = train_test_split(
    X,y,test_size=0.20, stratify=y, random_state=42)
# SECOND CUT: Split the remaining 20% into Validation and Test (50/50)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [5]:
print("Train set ",X_train.shape)
print("Train set ",X_val.shape)
print("Train set ",X_test.shape)

Train set  (8000, 6)
Train set  (1000, 6)
Train set  (1000, 6)


In [6]:
# First split, then fit the encoder only on the training set.
# ⚠️ Remember, to avoid "cheating" (data leakage), you Fit on the Training set and only Transform the others.

encoder = OrdinalEncoder(categories=[["low", "moderate", "high"]])

X_train["physical_activity_level"] = encoder.fit_transform(
    X_train[["physical_activity_level"]]
)

X_val["physical_activity_level"] = encoder.transform(
    X_val[["physical_activity_level"]]
)

X_test["physical_activity_level"] = encoder.transform(
    X_test[["physical_activity_level"]]
)

In [7]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

In [8]:
# Parameter distributions for randomized search
param_dist = {
    "n_estimators": [50, 100, 150, 200, 300],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10]
}



In [9]:
# randomized search
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [10]:
# 7. Fit on training set only
random_search.fit(X_train,y_train)
# 8. Best model
best_model = random_search.best_estimator_

print("Best parameters:")
print(random_search.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters:
{'subsample': 0.9, 'reg_lambda': 5, 'reg_alpha': 0, 'n_estimators': 50, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}


In [11]:
y_train_pred = best_model.predict(X_train)


In [12]:
y_val_pred = best_model.predict(X_val)

In [13]:
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred))

Train Accuracy: 0.767
Validation Accuracy: 0.742

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.91      0.83       700
           1       0.63      0.34      0.44       300

    accuracy                           0.74      1000
   macro avg       0.70      0.63      0.64      1000
weighted avg       0.72      0.74      0.72      1000



In [14]:
y_test_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.755

Test Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.92      0.84       700
           1       0.67      0.37      0.47       300

    accuracy                           0.76      1000
   macro avg       0.72      0.64      0.66      1000
weighted avg       0.74      0.76      0.73      1000

